# Fabric Error Framework — Usage Guide

**Notebook Resources required:** `fabric_error_codes.py`, `fabric_error_framework.py`

This notebook demonstrates every public API surface of the error framework:

| Section | What it covers |
|---------|----------------|
| 1. Setup | `sys.path` bootstrap, module import, env configuration |
| 2. Basic `handle_error` | Wrapping a simple step in try/except |
| 3. `retry_on_transient` | Decorator usage for flaky source reads |
| 4. Severity routing | How CRITICAL vs HIGH vs MEDIUM behave differently |
| 5. Full medallion pipeline | Bronze → Silver → Gold with per-layer error handling |
| 6. Querying the error log | Reading the Delta error table for observability |


## 1. Setup

Run this cell **first** in every notebook that uses the framework.
It adds the Notebook Resource folder to `sys.path` so Python can resolve
`fabric_error_codes` as a sibling module of `fabric_error_framework`.


In [ ]:
# ── 1a. Bootstrap sys.path ────────────────────────────────────────────────
import sys

# notebookutils.nbResPath resolves to the Notebook Resource folder at runtime.
# The try/except keeps the cell runnable in unit-test contexts outside Fabric.
try:
    _res_path = notebookutils.nbResPath
    if _res_path not in sys.path:
        sys.path.insert(0, _res_path)
    print(f'Resource path registered: {_res_path}')
except NameError:
    print('notebookutils not available — running outside Fabric runtime')


In [ ]:
# ── 1b. Import framework ──────────────────────────────────────────────────
import fabric_error_framework as ef

# ── 1c. Configure module-level constants ──────────────────────────────────
# Override the defaults so every FabricError carries correct context metadata.
ef.NOTEBOOK_NAME   = notebookutils.runtime.context['notebookName']
ef.ENVIRONMENT     = 'dev'          # Replace with 'prod' in production pipelines
ef.ERROR_TABLE_NAME = 'notebook_error_log'  # Managed Delta table in default lakehouse

print(f'Framework loaded  : fabric_error_framework')
print(f'Notebook name     : {ef.NOTEBOOK_NAME}')
print(f'Environment       : {ef.ENVIRONMENT}')
print(f'Error table       : {ef.ERROR_TABLE_NAME}')
print(f'Run ID            : {ef.RUN_ID}')


## 2. Basic `handle_error` Usage

> **Pattern:** wrap every distinct pipeline step in its own `try/except`.
> Pass the appropriate `ErrorCode` so the error log is queryable by category.


In [ ]:
# ── 2a. MEDIUM severity — non-fatal, pipeline continues ───────────────────
# Simulates a data type cast failure that can be tolerated.
try:
    raise ValueError('Cannot cast column [sale_amount] from STRING to DOUBLE — nulls present')
except Exception as ex:
    err = ef.handle_error(
        spark,
        ef.ErrorCode.TRN_2002,
        ex,
        cell_name='cast_sale_amount',
        record_count_affected=142,
        raise_on_critical=False,   # MEDIUM — let the pipeline continue
    )
    print(f'Handled [{err.code}] severity={err.severity} — continuing pipeline')


In [ ]:
# ── 2b. HIGH severity — step failed, pipeline may continue ────────────────
# Simulates a row count check that exceeds the failure threshold.
try:
    source_count, target_count = 10_000, 8_500
    if target_count < source_count * 0.95:
        raise AssertionError(
            f'Row count check failed: source={source_count}, target={target_count} '
            f'({target_count/source_count:.1%} < 95% threshold)'
        )
except Exception as ex:
    err = ef.handle_error(
        spark,
        ef.ErrorCode.VAL_3000,
        ex,
        cell_name='row_count_validation',
        record_count_affected=source_count - target_count,
        raise_on_critical=False,
    )
    print(f'Handled [{err.code}] severity={err.severity}')


In [ ]:
# ── 2c. CRITICAL severity — pipeline halts ────────────────────────────────
# Simulates a Key Vault secret retrieval failure.
# raise_on_critical=True (default) will re-raise, stopping the pipeline.
try:
    try:
        raise PermissionError('Access denied to secret [kv-adls-sas-token] — managed identity not assigned')
    except Exception as ex:
        ef.handle_error(
            spark,
            ef.ErrorCode.SEC_7000,
            ex,
            cell_name='get_kv_secret',
            raise_on_critical=True,   # Default — CRITICAL always re-raises
        )
except PermissionError as halted:
    print(f'Pipeline halted as expected: {halted}')


## 3. `retry_on_transient` Decorator

Use this decorator on any function that reads from an external source
where transient failures (timeouts, network blips) are expected.
The decorator retries with **linear backoff** before giving up.


In [ ]:
# Simulates a flaky source read that succeeds on the 3rd attempt.
_attempt_counter = 0


@ef.retry_on_transient(max_retries=3, delay_seconds=1)
def read_source_file(path: str):
    """Read raw CSV from ADLS — retries on transient I/O errors."""
    global _attempt_counter
    _attempt_counter += 1
    print(f'  read_source_file — attempt {_attempt_counter}')
    if _attempt_counter < 3:
        raise IOError(f'Connection reset by peer (attempt {_attempt_counter})')
    # Simulate successful read on attempt 3
    return spark.range(100).toDF('id')


try:
    df = read_source_file('abfss://bronze@onelake.dfs.fabric.microsoft.com/raw/sales.csv')
    print(f'Read succeeded — {df.count()} rows')
except Exception as ex:
    ef.handle_error(spark, ef.ErrorCode.SRC_1003, ex, cell_name='read_source_file')


In [ ]:
# Decorator + handle_error combined — all retries exhausted.
# Shows the full failure path logged to the Delta error table.
_fail_counter = 0


@ef.retry_on_transient(max_retries=2, delay_seconds=1)
def call_external_api(endpoint: str):
    global _fail_counter
    _fail_counter += 1
    raise ConnectionError(f'API unreachable after {_fail_counter} attempt(s): {endpoint}')


try:
    call_external_api('https://api.example.com/v1/prices')
except Exception as ex:
    ef.handle_error(
        spark,
        ef.ErrorCode.NET_6000,
        ex,
        cell_name='call_external_api',
        raise_on_critical=False,   # HIGH — log and continue
    )


## 4. Severity Routing Reference

The table below summarises the four severity levels and the recommended
`raise_on_critical` setting for each.

| Severity | `raise_on_critical` | Pipeline effect | Typical codes |
|----------|---------------------|-----------------|---------------|
| CRITICAL | `True` (default)    | Halts immediately | SRC-1000, SNK-4000, SEC-7000, SYS-800x |
| HIGH     | `False`             | Step skipped, pipeline continues | SRC-1001, VAL-3000, NET-6000 |
| MEDIUM   | `False`             | Logged, data quality warning | TRN-2002, VAL-3002 |
| LOW      | `False`             | Informational only | ALT-9000 |


In [ ]:
# Iterate all four severity levels for a quick visual reference.
demo_cases = [
    (ef.ErrorCode.SYS_8002, 'Simulated system crash',   False),
    (ef.ErrorCode.NET_6001, 'Simulated timeout',        False),
    (ef.ErrorCode.TRN_2003, 'Nulls exceed threshold',   False),
    (ef.ErrorCode.ALT_9000, 'Teams webhook unreachable', False),
]

for error_code, msg, raise_flag in demo_cases:
    try:
        raise RuntimeError(msg)
    except Exception as ex:
        err = ef.handle_error(
            spark, error_code, ex,
            cell_name='severity_routing_demo',
            raise_on_critical=raise_flag,
        )
        print(f'  [{err.code}]  severity={err.severity:<8}  description={err.error_code.description}')


## 5. Full Medallion Pipeline Example

Demonstrates the recommended per-layer error handling pattern
across Bronze → Silver → Gold, including a Delta `MERGE` at the Gold layer.

> Each layer function is decorated with `@retry_on_transient` for source reads
> and wraps its internal logic in `try/except` mapped to a specific `ErrorCode`.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime

# ── Shared schema ─────────────────────────────────────────────────────────
SALES_SCHEMA = StructType([
    StructField('order_id',    StringType(),    False),
    StructField('customer_id', StringType(),    True),
    StructField('product_id',  StringType(),    True),
    StructField('amount',      DoubleType(),    True),
    StructField('order_ts',    TimestampType(), True),
])

BRONZE_TABLE = 'bronze_sales_raw'
SILVER_TABLE = 'silver_sales_cleansed'
GOLD_TABLE   = 'gold_sales_summary'


In [ ]:
# ── BRONZE: Raw ingestion ─────────────────────────────────────────────────
@ef.retry_on_transient(max_retries=3, delay_seconds=5)
def ingest_bronze(source_path: str):
    """
    Read raw sales CSV from ADLS and append to the Bronze Delta table.
    Retries on transient I/O errors. Schema is enforced at this layer.
    """
    try:
        df_raw = (
            spark.read
            .schema(SALES_SCHEMA)
            .option('header', 'true')
            .csv(source_path)
        )

        # Validate the file was not empty
        row_count = df_raw.count()
        if row_count == 0:
            raise ValueError(f'Source file is empty: {source_path}')

        # Stamp ingestion metadata
        df_bronze = df_raw.withColumn('ingested_at', F.current_timestamp())

        df_bronze.write.format('delta').mode('append').saveAsTable(BRONZE_TABLE)
        print(f'Bronze: {row_count:,} rows ingested from {source_path}')
        return row_count

    except ValueError as ex:
        ef.handle_error(spark, ef.ErrorCode.SRC_1002, ex, cell_name='ingest_bronze')
    except Exception as ex:
        ef.handle_error(spark, ef.ErrorCode.SRC_1000, ex, cell_name='ingest_bronze')


In [ ]:
# ── SILVER: Cleansing & conforming ────────────────────────────────────────
def transform_silver():
    """
    Read Bronze, apply cleansing rules, write to Silver.
    Validates null thresholds and duplicate keys before writing.
    """
    try:
        df_bronze = spark.table(BRONZE_TABLE)
        total     = df_bronze.count()

        # ── Rule 1: Drop rows where amount is null or negative
        df_clean = df_bronze.filter(F.col('amount').isNotNull() & (F.col('amount') > 0))
        dropped  = total - df_clean.count()
        null_pct = dropped / total if total > 0 else 0

        if null_pct > 0.10:   # Fail if > 10 % of rows are invalid
            raise ValueError(
                f'Null/negative amount rows ({null_pct:.1%}) exceed 10% threshold'
            )
        elif null_pct > 0.02:
            ef.handle_error(
                spark,
                ef.ErrorCode.TRN_2003,
                ValueError(f'Null amount rows at {null_pct:.1%} — within tolerance'),
                cell_name='transform_silver',
                record_count_affected=dropped,
                raise_on_critical=False,
            )

        # ── Rule 2: Deduplicate on order_id (keep latest ingested_at)
        from pyspark.sql.window import Window
        window = Window.partitionBy('order_id').orderBy(F.col('ingested_at').desc())
        df_deduped = (
            df_clean
            .withColumn('_rn', F.row_number().over(window))
            .filter(F.col('_rn') == 1)
            .drop('_rn')
        )

        # ── Rule 3: Standardise types and add Silver metadata
        df_silver = (
            df_deduped
            .withColumn('amount',      F.round(F.col('amount'), 2))
            .withColumn('order_date',  F.to_date(F.col('order_ts')))
            .withColumn('cleansed_at', F.current_timestamp())
            .drop('ingested_at')
        )

        df_silver.write.format('delta').mode('overwrite').saveAsTable(SILVER_TABLE)
        print(f'Silver: {df_silver.count():,} cleansed rows written to {SILVER_TABLE}')

    except ValueError as ex:
        ef.handle_error(spark, ef.ErrorCode.VAL_3000, ex, cell_name='transform_silver')
    except Exception as ex:
        ef.handle_error(spark, ef.ErrorCode.TRN_2000, ex, cell_name='transform_silver')


In [ ]:
# ── GOLD: Aggregation + Delta MERGE ──────────────────────────────────────
def aggregate_gold():
    """
    Aggregate Silver to daily sales summary and MERGE into the Gold table.
    Uses Delta MERGE to make the operation idempotent.
    """
    try:
        from delta.tables import DeltaTable

        df_silver = spark.table(SILVER_TABLE)

        df_gold = (
            df_silver
            .groupBy('order_date', 'product_id')
            .agg(
                F.sum('amount').alias('total_sales'),
                F.count('order_id').alias('order_count'),
                F.avg('amount').alias('avg_order_value'),
            )
            .withColumn('aggregated_at', F.current_timestamp())
        )

        # Idempotent MERGE — safe to re-run
        if DeltaTable.isDeltaTable(spark, f'Tables/{GOLD_TABLE}'):
            DeltaTable.forName(spark, GOLD_TABLE).alias('target').merge(
                df_gold.alias('source'),
                'target.order_date = source.order_date AND target.product_id = source.product_id',
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            print(f'Gold: MERGE complete on {GOLD_TABLE}')
        else:
            df_gold.write.format('delta').mode('overwrite').saveAsTable(GOLD_TABLE)
            print(f'Gold: Initial load complete — {df_gold.count():,} rows written to {GOLD_TABLE}')

    except Exception as ex:
        ef.handle_error(spark, ef.ErrorCode.SNK_4001, ex, cell_name='aggregate_gold')


In [ ]:
# ── Orchestrate the full pipeline ─────────────────────────────────────────
SOURCE_PATH = 'abfss://raw@onelake.dfs.fabric.microsoft.com/sales/2025/01/sales_20250101.csv'

print('Starting pipeline run:', ef.RUN_ID)
print('─' * 60)

bronze_count = ingest_bronze(SOURCE_PATH)
transform_silver()
aggregate_gold()

print('─' * 60)
print(f'Pipeline complete  : {ef.RUN_ID}')


## 6. Querying the Error Log

The framework writes every handled error to a managed Delta table.
Use these queries for observability, alerting triage, and audit trails.


In [ ]:
# ── All errors for this notebook run ─────────────────────────────────────
df_errors = spark.table(ef.ERROR_TABLE_NAME)

display(
    df_errors
    .filter(F.col('run_id') == ef.RUN_ID)
    .orderBy('timestamp')
    .select('timestamp', 'error_code', 'severity', 'cell_name',
            'record_count_affected', 'message')
)


In [ ]:
# ── CRITICAL errors in the last 24 hours ─────────────────────────────────
display(
    df_errors
    .filter(
        (F.col('severity') == 'CRITICAL') &
        (F.col('timestamp') >= F.date_sub(F.current_date(), 1))
    )
    .orderBy(F.col('timestamp').desc())
    .select('timestamp', 'notebook_name', 'environment', 'error_code',
            'cell_name', 'message', 'run_id')
)


In [ ]:
# ── Error frequency by code — useful for trending / alerting thresholds ──
display(
    df_errors
    .groupBy('error_code', 'severity', 'environment')
    .agg(
        F.count('*').alias('occurrences'),
        F.sum('record_count_affected').alias('total_records_affected'),
        F.max('timestamp').alias('last_seen'),
    )
    .orderBy(F.col('occurrences').desc())
)
